# Cycle 1 — Tuning (Chronological Split)

Same `RandomizedSearchCV` configuration as `notebooks/cycle1_tuning.ipynb`. Only the train/test split is changed to chronological. Cross-validation is still `StratifiedKFold(n_splits=5)` over the **training** portion (which is itself contiguous in time) so the inner CV remains a defensible model-selection procedure on past data.

## Setup & data

In [2]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
from xgboost import XGBClassifier

# Dataset 1 -- chronological by Season
df1 = pd.read_csv('../../../data/processed/premier_league_matches_processed.csv').sort_values('Season').reset_index(drop=True)
split_idx1 = int(len(df1) * 0.8)
X1_train, y1_train = df1.iloc[:split_idx1].drop(columns=['FTR']), df1.iloc[:split_idx1]['FTR']
X1_test,  y1_test  = df1.iloc[split_idx1:].drop(columns=['FTR']), df1.iloc[split_idx1:]['FTR']

# Dataset 2 -- chronological by date
df2 = pd.read_csv('../../../data/processed/skysports_match_stats_processed.csv')
df2['date'] = pd.to_datetime(df2['date'])
df2 = df2.sort_values('date').reset_index(drop=True)
split_idx2 = int(len(df2) * 0.8)
X2_train, y2_train = df2.iloc[:split_idx2].drop(columns=['FTR','date']), df2.iloc[:split_idx2]['FTR']
X2_test,  y2_test  = df2.iloc[split_idx2:].drop(columns=['FTR','date']), df2.iloc[split_idx2:]['FTR']

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print(f'Dataset 1 -- Train: {len(X1_train)} | Test: {len(X1_test)}')
print(f'Dataset 2 -- Train: {len(X2_train)} | Test: {len(X2_test)}')

Dataset 1 -- Train: 5472 | Test: 1368
Dataset 2 -- Train: 896 | Test: 225


## XGBoost search — Dataset 1

In [3]:
xgb_param_grid = {
    'n_estimators':     [100,200,300,500],
    'max_depth':        [3,4,5,6],
    'learning_rate':    [0.01,0.05,0.1,0.2],
    'subsample':        [0.7,0.8,1.0],
    'colsample_bytree': [0.7,0.8,1.0],
    'min_child_weight': [1,3,5],
    'gamma':            [0,0.1,0.2],
}

xgb1 = XGBClassifier(random_state=42, eval_metric='mlogloss', verbosity=0)
search_d1 = RandomizedSearchCV(xgb1, xgb_param_grid, n_iter=50, cv=cv,
                                scoring='accuracy', random_state=42, n_jobs=-1, verbose=1)
search_d1.fit(X1_train, y1_train)

print('Best params:', search_d1.best_params_)
print(f'Best CV accuracy: {search_d1.best_score_*100:.2f}%')
y_pred_xgb_d1 = search_d1.best_estimator_.predict(X1_test)
print(f'Test accuracy:    {accuracy_score(y1_test, y_pred_xgb_d1)*100:.2f}%')
print()
print(classification_report(y1_test, y_pred_xgb_d1, target_names=['Away Win','Draw','Home Win']))

Fitting 5 folds for each of 50 candidates, totalling 250 fits
Best params: {'subsample': 0.8, 'n_estimators': 100, 'min_child_weight': 3, 'max_depth': 4, 'learning_rate': 0.05, 'gamma': 0.1, 'colsample_bytree': 0.7}
Best CV accuracy: 53.42%
Test accuracy:    52.34%

              precision    recall  f1-score   support

    Away Win       0.52      0.44      0.48       406
        Draw       0.30      0.03      0.05       348
    Home Win       0.53      0.86      0.66       614

    accuracy                           0.52      1368
   macro avg       0.45      0.44      0.40      1368
weighted avg       0.47      0.52      0.45      1368



## XGBoost search — Dataset 2

In [4]:
xgb2 = XGBClassifier(random_state=42, eval_metric='mlogloss', verbosity=0)
search_d2 = RandomizedSearchCV(xgb2, xgb_param_grid, n_iter=50, cv=cv,
                                scoring='accuracy', random_state=42, n_jobs=-1, verbose=1)
search_d2.fit(X2_train, y2_train)

print('Best params:', search_d2.best_params_)
print(f'Best CV accuracy: {search_d2.best_score_*100:.2f}%')
y_pred_xgb_d2 = search_d2.best_estimator_.predict(X2_test)
print(f'Test accuracy:    {accuracy_score(y2_test, y_pred_xgb_d2)*100:.2f}%')
print()
print(classification_report(y2_test, y_pred_xgb_d2, target_names=['Away Win','Draw','Home Win']))

Fitting 5 folds for each of 50 candidates, totalling 250 fits
Best params: {'subsample': 1.0, 'n_estimators': 300, 'min_child_weight': 1, 'max_depth': 3, 'learning_rate': 0.01, 'gamma': 0.2, 'colsample_bytree': 0.8}
Best CV accuracy: 53.12%
Test accuracy:    50.22%

              precision    recall  f1-score   support

    Away Win       0.42      0.52      0.46        64
        Draw       0.30      0.12      0.17        51
    Home Win       0.58      0.67      0.62       110

    accuracy                           0.50       225
   macro avg       0.44      0.44      0.42       225
weighted avg       0.47      0.50      0.48       225



## Random Forest search — Dataset 2

In [5]:
rf_param_grid = {
    'n_estimators':     [100,200,300,500],
    'max_depth':        [None,5,10,15,20],
    'min_samples_split':[2,5,10],
    'min_samples_leaf': [1,2,4],
    'max_features':     ['sqrt','log2',None],
    'class_weight':     ['balanced','balanced_subsample'],
}

rf = RandomForestClassifier(random_state=42)
search_rf_d2 = RandomizedSearchCV(rf, rf_param_grid, n_iter=50, cv=cv,
                                   scoring='accuracy', random_state=42, n_jobs=-1, verbose=1)
search_rf_d2.fit(X2_train, y2_train)

print('Best params:', search_rf_d2.best_params_)
print(f'Best CV accuracy: {search_rf_d2.best_score_*100:.2f}%')
y_pred_rf_d2 = search_rf_d2.best_estimator_.predict(X2_test)
print(f'Test accuracy:    {accuracy_score(y2_test, y_pred_rf_d2)*100:.2f}%')

Fitting 5 folds for each of 50 candidates, totalling 250 fits
Best params: {'n_estimators': 200, 'min_samples_split': 5, 'min_samples_leaf': 1, 'max_features': 'log2', 'max_depth': None, 'class_weight': 'balanced_subsample'}
Best CV accuracy: 50.44%
Test accuracy:    50.22%


## Side-by-side comparison

Random-split tuned numbers come from `notebooks/cycle1/cycle1_tuning.ipynb` (Pipeline-based, honest per-fold CV): D1 XGB **52.78%**, D2 XGB **46.22%**, D2 RF **44.44%**.

**Bottom line: chronological tuning beats honest random tuning on Dataset 2.** D2 XGB Tuned moves from 46.22% (random) to 50.22% (chronological); D2 RF Tuned from 44.44% to 50.22%.

In [6]:
comp = pd.DataFrame([
    {'Dataset':'Dataset 1','Model':'XGBoost Tuned',       'Chrono':accuracy_score(y1_test,y_pred_xgb_d1)*100,'Random':52.78},
    {'Dataset':'Dataset 2','Model':'XGBoost Tuned',       'Chrono':accuracy_score(y2_test,y_pred_xgb_d2)*100,'Random':46.22},
    {'Dataset':'Dataset 2','Model':'Random Forest Tuned', 'Chrono':accuracy_score(y2_test,y_pred_rf_d2)*100, 'Random':44.44},
])
comp['Delta'] = (comp['Chrono']-comp['Random']).round(2)
comp['Chrono'] = comp['Chrono'].round(2)
print(comp.to_string(index=False))

  Dataset               Model  Chrono  Random  Delta
Dataset 1       XGBoost Tuned   52.34   52.78  -0.44
Dataset 2       XGBoost Tuned   50.22   46.22   4.00
Dataset 2 Random Forest Tuned   50.22   44.44   5.78
